In [30]:
import numpy as np
import pandas as pd
import warnings

from tqdm import tqdm, TqdmWarning
from datetime import datetime
from pathlib import Path
from typing import Literal


warnings.filterwarnings("ignore", category=TqdmWarning)

In [31]:
Type = Literal["train", "test"]
Split = Literal["split_01", "split_02", "split_03", "split_04", "split_05", "split_06", "split_07", "split_08", "split_09", "split_10", "split_11", "split_12", "split_13", "split_14", "split_15", "split_16", "split_17", "split_18", "split_19", "split_20"]


EPS = np.finfo(float).eps


def now() -> str:
    return datetime.now().astimezone().strftime("%Y%m%d-%H%M%S-%z")

### Data Loading

In [32]:
def load_log_df(type: Type, **kwargs) -> pd.DataFrame:
    return pd.read_parquet(f"../artifacts/kaggle/{type}_log.parquet", **kwargs)

def load_flc_df(type: Type, split: Split, **kwargs) -> pd.DataFrame:
    return pd.read_parquet(f"../artifacts/kaggle/{split}/{type}_flc.parquet", **kwargs)


def __ingest_dfs(type: Type):
    log_df = pd.read_csv(f"../artifacts/kaggle/{type}_log.csv", index_col="object_id")
    log_df.to_parquet(f"../artifacts/kaggle/{type}_log.parquet")

    splits = sorted(log_df["split"].unique())
    for split in splits:
        flc_df = pd.read_csv(f"../artifacts/kaggle/{split}/{type}_full_lightcurves.csv")
        flc_df.to_parquet(f"../artifacts/kaggle/{split}/{type}_flc.parquet")


__ingest_dfs(type="train")
__ingest_dfs(type="test")

### Feature Engineering

In [33]:
def load_feats_df(type: Type, split: Split, **kwargs):
    return pd.read_parquet(f"../artifacts/feats/{split}/{type}_feats.parquet", **kwargs)


def __ingest_feats(type: Type):
    log_df = load_log_df(type=type)

    with tqdm(total=len(log_df), unit="obj") as pb:
        for split, log_sub_df in log_df.groupby("split"):
            pb.set_description(f"Loading features for `{split}` (`{type}`)")

            feats_buf = []

            for obj_id, log_row in log_sub_df.iterrows():
                flc_sub_df = load_flc_df(type=type, split=split, filters=[("object_id", "==", obj_id)]) # type: ignore
                feats = __load_feats_for_obj(log_row, flc_sub_df)
                feats_buf.append(feats)

                pb.update()

            pb.set_description(f"Ingesting features for `{split}` (`{type}`)")

            Path(f"../artifacts/feats/{split}").mkdir(parents=True, exist_ok=True)
            pd.DataFrame(feats_buf).to_parquet(f"../artifacts/feats/{split}/{type}_feats.parquet")


def __load_feats_for_obj(log_row: pd.Series, flc_sub_df: pd.DataFrame) -> dict:
    flc_sub_df["Flux_ratio"] = flc_sub_df["Flux"] / flc_sub_df["Flux_err"]
    pivot_flc_sub_df = flc_sub_df.pivot_table(index="Time (MJD)", columns="Filter", values=["Flux", "Flux_err", "Flux_ratio"])

    feats = log_row.to_dict()

    def q25(series: pd.Series): return series.quantile(0.25)
    def q75(series: pd.Series): return series.quantile(0.75)
    def skew(series: pd.Series): return series.skew()
    def kurt(series: pd.Series): return series.kurt()

    for agg in [np.mean, np.std, np.min, np.max, np.median, q25, q75, skew, kurt]:
        for feat in ["Flux", "Flux_err", "Flux_ratio"]:
            feats[f"{feat}_{agg.__name__}"] = agg(flc_sub_df[feat])
            
            for filter in ["u", "g", "r", "i", "z", "y"]:
                feats[f"{feat}_{agg.__name__}_{filter}"] = agg(pivot_flc_sub_df[(feat, filter)]) if filter in pivot_flc_sub_df.columns else np.nan

    return feats


__ingest_feats(type="train")
# __ingest_feats(type="test")


# def de_extinct(log_df: pd.DataFrame, flc_df: pd.DataFrame):
#     EXTINCTION_COEFFS = {
#         "u": 4.81,
#         "g": 3.64,
#         "r": 2.70,
#         "i": 2.06,
#         "z": 1.58,
#         "y": 1.31
#     }

#     ebv = flc_df["object_id"].map(log_df["EBV"])
#     r_λ = flc_df["Filter"].map(EXTINCTION_COEFFS)

#     c_λ = np.pow(10, 0.4 * r_λ * ebv)

#     flc_df["Flux"] *= c_λ
#     flc_df["Flux_err"] *= c_λ


# TODO Consider extinction.fitzpatrick99

Ingesting features for `split_20` (`train`): 100%|██████████| 3043/3043 [00:25<00:00, 118.92obj/s]


In [34]:
feats = load_feats_df(type="train", split="split_01")

feats

,Z,Z_err,EBV,SpecType,English Translation,split,target,Flux_mean,Flux_mean_u,Flux_mean_g,...,Flux_err_kurt_i,Flux_err_kurt_z,Flux_err_kurt_y,Flux_ratio_kurt,Flux_ratio_kurt_u,Flux_ratio_kurt_g,Flux_ratio_kurt_r,Flux_ratio_kurt_i,Flux_ratio_kurt_z,Flux_ratio_kurt_y
0,3.0490,NaN,0.110,AGN,Trawn Folk (Dwarfs) + northern + Ents (people),split_01,0,0.928483,NaN,NaN,...,NaN,NaN,NaN,15.455500,NaN,NaN,NaN,NaN,NaN,NaN
1,0.4324,NaN,0.058,SN II,Trawn Folk (Dwarfs) + tree + drinking vessel,split_01,0,0.388622,NaN,NaN,...,NaN,NaN,NaN,15.072513,NaN,NaN,NaN,NaN,NaN,NaN
2,0.4673,NaN,0.577,AGN,Elves + lover (fem.) + breath,split_01,0,1.691347,NaN,NaN,...,NaN,NaN,NaN,0.024516,NaN,NaN,NaN,NaN,NaN,NaN
3,0.6946,NaN,0.012,AGN,moon + roof + noble maiden,split_01,0,0.375366,NaN,NaN,...,NaN,NaN,NaN,3.230106,NaN,NaN,NaN,NaN,NaN,NaN
4,0.4161,NaN,0.058,AGN,"jewel, Silmaril + father + Wild Man",split_01,0,0.233832,NaN,NaN,...,NaN,NaN,NaN,2.120613,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150,0.2758,NaN,0.064,TDE,"roof + Elves + broth, liquid food, soup",split_01,1,0.210834,NaN,NaN,...,NaN,NaN,NaN,54.000375,NaN,NaN,NaN,NaN,NaN,NaN
151,0.3751,NaN,0.013,AGN,"roof + harbour, haven, small landlocked bay ...",split_01,0,1.865482,NaN,NaN,...,NaN,NaN,NaN,1.094542,NaN,NaN,NaN,NaN,NaN,NaN
152,0.9238,NaN,0.031,AGN,eternity + flowering + water-fall,split_01,0,0.496488,NaN,NaN,...,NaN,NaN,NaN,0.380703,NaN,NaN,NaN,NaN,NaN,NaN
153,0.3303,NaN,0.043,AGN,"evil creature + beacon, fire-sign + lake, pool",split_01,0,0.285122,NaN,NaN,...,NaN,NaN,NaN,1.737381,NaN,NaN,NaN,NaN,NaN,NaN


### Training

In [35]:
# __dirpath = Path("../artifacts/predictions")
# __dirpath.mkdir(parents=True, exist_ok=True)

# prediction_df.to_csv(f"{__dirpath}/submission-{now()}.csv", index=False)